# Flickr30K Captioning Evaluation — baseline vs AttnRes (3 seeds)

Evaluates the frozen-SigLIP → linear connector → GPT decoder VLM
(`SiglipAttnResCaptioner`) on the Flickr30K captioning split, comparing the
`baseline` and `attnres` decoder variants across seeds **123 / 456 / 789**.

**Pipeline stages (one section each):**
1. Dependency setup (`pycocoevalcap`, Java for METEOR/SPICE)
2. Repo discovery + imports (reuses the existing codebase)
3. Config
4. `Evaluator` interface + `CaptioningEvaluator`
5. Checkpoint discovery (Google Drive, optional W&B fallback)
6. Model loading + per-seed eval data
7. Generation + on-disk caching
8. Scoring + aggregation (mean ± std)
9. Results table (CIDEr first)

**Reuses existing logic** — checkpoints are loaded exactly like the training
script (`torch.load(...)[\"model\"]` into a freshly built `SiglipAttnResCaptioner`),
and the eval split is rebuilt with `src.vlm.flickr30k.load_flickr30k_examples`
using the **same seed and `max_examples`** so each checkpoint is scored on the
held-out images it never trained on (the 90/10 split is seed-dependent).

> Generation strategy: **greedy**. References: **all human captions per val image**.
> Designed to run on Colab (T4) with checkpoints on Google Drive.

## 1. Dependency setup

Installs `pycocoevalcap` (CIDEr, BLEU-4, METEOR, ROUGE-L, SPICE). METEOR, SPICE
and the PTB tokenizer need a JVM — on Colab we try to install one. If Java is
unavailable we fall back to a simple tokenizer and skip METEOR/SPICE.

In [1]:
import shutil
import subprocess
import sys


def _pip_install(*pkgs: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


try:
    import pycocoevalcap  # noqa: F401
except ImportError:
    _pip_install("pycocoevalcap")

# METEOR / SPICE / PTBTokenizer require a JVM. Best-effort install on Colab.
if shutil.which("java") is None:
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "default-jre"], check=True)
    except Exception as exc:  # noqa: BLE001
        print(f"Could not auto-install Java ({exc}); METEOR/SPICE will be skipped.")

# Pure-Python scorers are always available; import once for later cells.
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

JAVA_OK = shutil.which("java") is not None
print("pycocoevalcap OK | java available =", JAVA_OK)

pycocoevalcap OK | java available = True


## 2. Mount Google Drive (required)

Seed checkpoints live at `MyDrive/AttnResGPT-next-scale-artifacts/checkpoints/`.
The Colab login and Drive account **can differ** — in the Google popup click
**Use another account** to pick the account that owns the checkpoints.
When Colab asks for Drive permissions, enable **all** checkboxes (read-only often fails).

In [2]:
from pathlib import Path

DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"
DRIVE_MOUNT = Path("/content/drive")
DRIVE_CHECKPOINTS = None


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if not _in_colab():
    raise RuntimeError("Run on Google Colab — VLM seed checkpoints are stored on Google Drive.")

from google.colab import auth, drive

print(
    "Mount the Google account that owns the checkpoint folder.\n"
    "Tip: click 'Use another account' if it defaults to your Colab login.\n"
    "Tip: enable ALL Drive permission boxes in the popup."
)

for attempt, force in enumerate([False, True], start=1):
    try:
        auth.authenticate_user()
        drive.mount(str(DRIVE_MOUNT), force_remount=force)
        DRIVE_CHECKPOINTS = DRIVE_MOUNT / "MyDrive" / DRIVE_ARTIFACT_FOLDER / "checkpoints"
        print("Drive mounted:", DRIVE_MOUNT)
        print("Checkpoint dir:", DRIVE_CHECKPOINTS, "| exists =", DRIVE_CHECKPOINTS.exists())
        break
    except Exception as exc:
        if attempt == 2:
            raise RuntimeError(
                "Google Drive mount failed — cannot load seed checkpoints.\n"
                "Try: Runtime > Disconnect and delete runtime, allow popups for colab.research.google.com,\n"
                "enable ALL Drive permissions, sign into the account with the checkpoints.\n"
                "Known Colab issue: https://github.com/googlecolab/colabtools/issues/5944"
            ) from exc
        print(f"Mount attempt {attempt} failed: {exc}\nRetrying with force_remount=True ...")


Mounted at /content/drive
Drive mounted: /content/drive
Checkpoint dir: /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints | exists = True


## 3. Repo setup + imports

Find or clone the repo, install deps, import from `src`. Requires section 2 (Drive mounted).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path | None:
    for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/AttnResGPT-next-scale")]:
        if is_repo_root(candidate):
            return candidate.resolve()
    for root in [Path("/content"), Path("/content/drive/MyDrive")]:
        if not root.exists():
            continue
        for path in root.rglob("AttnResGPT-next-scale"):
            if is_repo_root(path):
                return path.resolve()
    return None


if DRIVE_CHECKPOINTS is None:
    raise RuntimeError("Run the Google Drive mount cell (section 2) first.")

REPO = discover_repo()
if REPO is None:
    target = Path("/content/AttnResGPT-next-scale")
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    REPO = target.resolve()

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["git", "fetch", "--quiet"], check=False)
subprocess.run(["git", "pull", "--ff-only"], check=False)
print("repo root:", REPO)

import torch
from torch.utils.data import DataLoader, Dataset

from src.data.tokenizer import build_tokenizer
from src.models.vlm_attnres import SiglipAttnResCaptioner
from src.utils.config import load_config
from src.vlm.flickr30k import _row_captions, _row_image, load_flickr30k_examples

print("codebase imports OK")


## 4. Config

All knobs in one place. `MAX_EXAMPLES`, `DATASET_NAME`, `TOKENIZER_NAME`,
`VISION_MODEL` and `DECODER_CONFIG` must match training so the seed-dependent
90/10 split is reproduced exactly. `VLM_BATCH_SIZE` only affects the checkpoint
folder name (`vlm_{variant}_flickr30k_b{batch}_seed{seed}`).

In [ ]:
SEEDS = [123, 456, 789]
VARIANTS = ["baseline", "attnres"]

# --- must match training (reproduces the split + rebuilds the model) ---
VISION_MODEL = "google/siglip-base-patch16-224"
DECODER_CONFIG = "configs/large_ctx512_3000.yaml"
TOKENIZER_NAME = "gpt2"
DATASET_NAME = "Mozilla/flickr30k-transformed-captions"
DATASET_SPLIT = "train"
MAX_EXAMPLES = 20000          # training default; drives the seed-dependent 90/10 split
DECODER_MAX_SEQ_LEN = 512
VLM_BATCH_SIZE = 8            # only affects checkpoint folder name
FINAL_CHECKPOINT = "step_0006750.pt"

# --- evaluation knobs ---
MAX_GEN_TOKENS = 40
GEN_BATCH_SIZE = 16
FORCE_REGEN = False           # True = ignore cached captions and re-generate

# --- optional W&B fallback for missing Drive checkpoints ---
ENABLE_WANDB_FALLBACK = False
WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale"

if DRIVE_CHECKPOINTS is None:
    raise RuntimeError("Run the Google Drive mount cell (section 2) first.")
CHECKPOINT_ROOTS = [DRIVE_CHECKPOINTS]
CACHE_DIR = REPO / "outputs" / "caption_eval"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("checkpoint roots:", [str(r) for r in CHECKPOINT_ROOTS])
print("cache dir:", CACHE_DIR)


## 5. Evaluator interface

`Evaluator` is an abstract strategy: `generate(model, dataloader)` produces
prediction records (each carrying its own references), `score(predictions)`
returns a metric dict, and `run` chains them. The generation/aggregation loop
below only ever calls these methods, so a future `VQAEvaluator` or
`RetrievalEvaluator` drops in as a new subclass with no pipeline changes.

`CaptioningEvaluator` implements greedy autoregressive decoding (the VLM has no
`.generate()`, so we roll our own) and scores with `pycocoevalcap`.

In [ ]:
from abc import ABC, abstractmethod
from typing import Any


class Evaluator(ABC):
    """Abstract evaluation strategy. Subclass per task (captioning, VQA, ...)."""

    name: str = "base"

    @abstractmethod
    def generate(self, model: Any, dataloader: Any) -> list[dict]:
        """Run `model` over `dataloader`; return per-example prediction records."""

    @abstractmethod
    def score(self, predictions: list[dict]) -> dict[str, float]:
        """Score prediction records (they carry their own references)."""

    def run(self, model: Any, dataloader: Any) -> dict[str, float]:
        return self.score(self.generate(model, dataloader))


class CaptioningEvaluator(Evaluator):
    """Greedy caption generation + pycocoevalcap scoring (CIDEr/BLEU/METEOR/ROUGE/SPICE)."""

    name = "captioning"

    def __init__(
        self,
        tokenizer=None,
        device: "torch.device | None" = None,
        *,
        max_gen_tokens: int = 40,
        max_seq_len: int = 512,
        java_ok: bool = False,
    ) -> None:
        self.tokenizer = tokenizer
        self.device = device
        self.max_gen_tokens = max_gen_tokens
        self.max_seq_len = max_seq_len
        self.java_ok = java_ok
        # Token ids are only needed for generation (not scoring).
        self.bos_id = self.eos_id = None
        if tokenizer is not None:
            backend = tokenizer.backend
            pad = backend.pad_token_id if backend.pad_token_id is not None else 0
            bos = backend.bos_token_id
            if bos is None:
                bos = backend.eos_token_id
            if bos is None:
                bos = pad
            self.bos_id = int(bos)
            self.eos_id = int(backend.eos_token_id) if backend.eos_token_id is not None else int(pad)

    @torch.no_grad()
    def generate(self, model, dataloader) -> list[dict]:
        if self.tokenizer is None:
            raise ValueError("CaptioningEvaluator needs a tokenizer to generate.")
        model.eval()
        records: list[dict] = []
        for batch in dataloader:
            pixel_values = batch["pixel_values"].to(self.device)
            vision_hidden = model.encode_vision(pixel_values)
            batch_size = pixel_values.size(0)
            prefix_len = vision_hidden.size(1)
            input_ids = torch.full((batch_size, 1), self.bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)
            budget = max(1, min(self.max_gen_tokens, self.max_seq_len - prefix_len - 1))
            for _ in range(budget):
                output = model(vision_hidden=vision_hidden, input_ids=input_ids, return_aux=False)
                next_token = output["logits"][:, -1, :].argmax(dim=-1)
                next_token = torch.where(finished, torch.full_like(next_token, self.eos_id), next_token)
                input_ids = torch.cat([input_ids, next_token.unsqueeze(1)], dim=1)
                finished = finished | (next_token == self.eos_id)
                if bool(finished.all()):
                    break
            generated = input_ids[:, 1:].tolist()  # drop the seeded BOS
            for row, image_id, references in zip(generated, batch["image_ids"], batch["references"]):
                tokens: list[int] = []
                for token in row:
                    if token == self.eos_id:
                        break
                    tokens.append(int(token))
                caption = self.tokenizer.backend.decode(tokens).strip()
                records.append(
                    {"image_id": int(image_id), "prediction": caption, "references": list(references)}
                )
        return records

    def _tokenize(self, gts: dict, res: dict) -> tuple[dict, dict]:
        if self.java_ok:
            from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

            ptb = PTBTokenizer()
            return ptb.tokenize(gts), ptb.tokenize(res)

        def simple(table: dict) -> dict:
            return {key: [entry["caption"].lower().strip() for entry in value] for key, value in table.items()}

        return simple(gts), simple(res)

    def score(self, predictions: list[dict]) -> dict[str, float]:
        gts = {
            p["image_id"]: [{"caption": r} for r in p["references"] if isinstance(r, str) and r.strip()]
            for p in predictions
        }
        res = {
            p["image_id"]: [{"caption": p["prediction"] if p["prediction"].strip() else "."}]
            for p in predictions
        }
        valid = [k for k in gts if gts[k]]
        gts = {k: gts[k] for k in valid}
        res = {k: res[k] for k in valid}
        if not gts:
            return {}

        gts_tok, res_tok = self._tokenize(gts, res)
        scorers: list[tuple[Any, Any]] = [
            (Cider(), "CIDEr"),
            (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
            (Rouge(), "ROUGE_L"),
        ]
        if self.java_ok:
            from pycocoevalcap.meteor.meteor import Meteor

            scorers.append((Meteor(), "METEOR"))
            try:
                from pycocoevalcap.spice.spice import Spice

                scorers.append((Spice(), "SPICE"))
            except Exception as exc:  # noqa: BLE001
                print(f"  (SPICE unavailable: {exc})")

        metrics: dict[str, float] = {}
        for scorer, names in scorers:
            try:
                score, _ = scorer.compute_score(gts_tok, res_tok)
            except Exception as exc:  # noqa: BLE001
                print(f"  (scorer {names} failed: {exc})")
                continue
            if isinstance(names, list):
                for name, value in zip(names, score):
                    metrics[name] = float(value)
            else:
                metrics[names] = float(score)
        return metrics


print("Evaluator + CaptioningEvaluator defined")

## 6. Checkpoint discovery

Looks for `vlm_{variant}_flickr30k_b{batch}_seed{seed}/step_0006750.pt` under each
checkpoint root (Drive first, then local), preferring the final step and falling
back to the latest available. Prints what it found so you can sanity-check before
generation. If `ENABLE_WANDB_FALLBACK` is set, missing checkpoints are pulled from
the W&B run artifact using the existing `src.analysis.attnres_wandb` helpers.

In [ ]:
import re

CKPT_RE = re.compile(r"step_(\d+)\.pt$")


def run_folder_name(variant: str, seed: int) -> str:
    return f"vlm_{variant}_flickr30k_b{VLM_BATCH_SIZE}_seed{seed}"


def discover_checkpoint(variant: str, seed: int) -> "Path | None":
    folder = run_folder_name(variant, seed)
    for root in CHECKPOINT_ROOTS:
        run_dir = root / folder
        if not run_dir.exists():
            continue
        preferred = run_dir / FINAL_CHECKPOINT
        if preferred.exists():
            return preferred
        checkpoints = sorted(
            run_dir.glob("step_*.pt"),
            key=lambda p: int(CKPT_RE.search(p.name).group(1)),
        )
        if checkpoints:
            return checkpoints[-1]
    return None


def wandb_fallback(variant: str, seed: int, target_dir: Path) -> Path:
    """Best-effort download of a missing checkpoint from its W&B run artifact."""
    from src.analysis.attnres_wandb import (
        download_checkpoint_from_artifact,
        find_logged_checkpoint_artifact,
        login_wandb_from_env,
    )

    api = login_wandb_from_env()
    run_name = run_folder_name(variant, seed)
    runs = list(api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}", filters={"display_name": run_name}))
    if not runs:
        raise FileNotFoundError(f"No W&B run named {run_name}")
    run = runs[0]
    artifact = find_logged_checkpoint_artifact(api, run_path=f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run.id}")
    target_dir.mkdir(parents=True, exist_ok=True)
    path, _step = download_checkpoint_from_artifact(artifact, step=None, target_dir=target_dir)
    return path


discovered: dict[tuple[str, int], "Path | None"] = {}
print(f"{'variant':9} {'seed':>5}  checkpoint")
print("-" * 78)
for variant in VARIANTS:
    for seed in SEEDS:
        ckpt = discover_checkpoint(variant, seed)
        discovered[(variant, seed)] = ckpt
        print(f"{variant:9} {seed:>5}  {ckpt if ckpt else 'MISSING'}")

missing = [k for k, v in discovered.items() if v is None]
if missing:
    print("\nMissing:", missing, "| W&B fallback:", ENABLE_WANDB_FALLBACK)

## 7. Model loading + per-seed eval data

`load_vlm` rebuilds `SiglipAttnResCaptioner` with the same decoder config used in
training and loads `checkpoint["model"]` (identical to the training script's
resume path). `build_caption_eval_loader` rebuilds the **seed-specific** 90/10
split via `load_flickr30k_examples` and enforces an **image-level holdout**: it
keeps only val images whose captions never appear in train (dropping the single
boundary image), then gathers **all** human captions per image as references.

In [ ]:
def build_vlm(variant: str):
    config = load_config(
        str(REPO / DECODER_CONFIG),
        overrides=[
            f"model.architecture={variant}",
            f"data.block_size={DECODER_MAX_SEQ_LEN}",
            f"model.max_seq_len={DECODER_MAX_SEQ_LEN}",
        ],
    )
    tokenizer = build_tokenizer(TOKENIZER_NAME)
    config.model.vocab_size = tokenizer.vocab_size
    model = SiglipAttnResCaptioner(vision_model_name=VISION_MODEL, decoder_config=config.model)
    return model, tokenizer


def load_vlm(variant: str, checkpoint_path: "str | Path"):
    model, tokenizer = build_vlm(variant)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE).eval()
    return model, tokenizer, checkpoint.get("step")


class CaptionGenDataset(Dataset):
    """Unique val images with all their reference captions."""

    def __init__(self, dataset, image_ids: list[int]) -> None:
        self.dataset = dataset
        self.image_ids = image_ids

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, index: int) -> dict:
        row_index = int(self.image_ids[index])
        row = self.dataset[row_index]
        return {"image_id": row_index, "image": _row_image(row), "references": _row_captions(row)}


class CaptionGenCollator:
    def __init__(self, processor) -> None:
        self.processor = processor

    def __call__(self, items: list[dict]) -> dict:
        images = [item["image"] for item in items]
        pixel_values = self.processor(images=images, return_tensors="pt")["pixel_values"]
        return {
            "pixel_values": pixel_values,
            "image_ids": [item["image_id"] for item in items],
            "references": [item["references"] for item in items],
        }


def build_caption_eval_loader(processor, seed: int, *, batch_size: int):
    dataset, train_records, val = load_flickr30k_examples(
        dataset_name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_examples=MAX_EXAMPLES,
        seed=seed,
    )
    # Enforce an image-level holdout: an image is a valid eval image only if NONE
    # of its captions appear in train. The underlying split is caption-level, so
    # this drops the single boundary image straddling the 90/10 cutoff, leaving a
    # leak-free set of images the model never saw during training.
    train_ids = {record.row_index for record in train_records}
    unique_ids: list[int] = []
    seen: set[int] = set()
    for record in val:
        row_index = record.row_index
        if row_index in train_ids or row_index in seen:
            continue
        seen.add(row_index)
        unique_ids.append(row_index)
    loader = DataLoader(
        CaptionGenDataset(dataset, unique_ids),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=CaptionGenCollator(processor),
    )
    return loader, len(unique_ids)


print("model loading + eval-data helpers defined")

## 8. Generation + caching

For each `(variant, seed)`: load the checkpoint, rebuild that seed's val loader,
greedily generate one caption per image, and cache `{image_id, prediction,
references}` to `outputs/caption_eval/{variant}_seed{seed}_captions.json`. Cached
runs are reused (set `FORCE_REGEN = True` to override). Any failure is logged and
the loop continues with the remaining checkpoints.

In [ ]:
import gc
import json
import traceback

all_predictions: dict[tuple[str, int], list[dict]] = {}
generation_log: list[tuple[str, int, str, int]] = []

for variant in VARIANTS:
    for seed in SEEDS:
        cache_path = CACHE_DIR / f"{variant}_seed{seed}_captions.json"
        try:
            if cache_path.exists() and not FORCE_REGEN:
                predictions = json.loads(cache_path.read_text())
                all_predictions[(variant, seed)] = predictions
                generation_log.append((variant, seed, "cached", len(predictions)))
                print(f"[cache] {variant} seed{seed}: {len(predictions)} captions")
                continue

            checkpoint_path = discovered.get((variant, seed))
            if checkpoint_path is None and ENABLE_WANDB_FALLBACK:
                try:
                    checkpoint_path = wandb_fallback(
                        variant, seed, REPO / "checkpoints" / run_folder_name(variant, seed)
                    )
                    print(f"[wandb] pulled checkpoint for {variant} seed{seed}: {checkpoint_path}")
                except Exception as exc:  # noqa: BLE001
                    print(f"[wandb] fallback failed for {variant} seed{seed}: {exc}")

            if checkpoint_path is None:
                generation_log.append((variant, seed, "missing_checkpoint", 0))
                print(f"[skip] {variant} seed{seed}: no checkpoint")
                continue

            model, tokenizer, step = load_vlm(variant, checkpoint_path)
            evaluator = CaptioningEvaluator(
                tokenizer,
                DEVICE,
                max_gen_tokens=MAX_GEN_TOKENS,
                max_seq_len=DECODER_MAX_SEQ_LEN,
                java_ok=JAVA_OK,
            )
            loader, n_images = build_caption_eval_loader(model.processor, seed, batch_size=GEN_BATCH_SIZE)
            print(f"[gen] {variant} seed{seed} (step={step}): generating for {n_images} val images ...")
            predictions = evaluator.generate(model, loader)
            cache_path.write_text(json.dumps(predictions, indent=2))
            all_predictions[(variant, seed)] = predictions
            generation_log.append((variant, seed, "generated", len(predictions)))
            print(f"[gen] {variant} seed{seed}: {len(predictions)} captions -> {cache_path.name}")

            del model, evaluator, loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as exc:  # noqa: BLE001
            generation_log.append((variant, seed, f"error: {exc}", 0))
            print(f"[error] {variant} seed{seed}: {exc}")
            traceback.print_exc()

print("\nGeneration summary")
print("-" * 50)
for variant, seed, status, count in generation_log:
    print(f"  {variant:9} seed{seed}: {status} ({count})")

## 9. Scoring + aggregation

Scores the cached captions per `(variant, seed)`, then aggregates across seeds:
mean ± sample std (ddof=1) for every metric. Scoring reads only from
`all_predictions`, so you can re-run this section without re-generating.

In [ ]:
import numpy as np

# Reused for scoring only (no model/tokenizer needed).
_scorer = CaptioningEvaluator(java_ok=JAVA_OK)

per_seed_metrics: dict[tuple[str, int], dict[str, float]] = {}
for (variant, seed), predictions in all_predictions.items():
    try:
        metrics = _scorer.score(predictions)
        per_seed_metrics[(variant, seed)] = metrics
        cider = metrics.get("CIDEr")
        print(f"[score] {variant} seed{seed}: CIDEr={cider:.4f}" if cider is not None
              else f"[score] {variant} seed{seed}: {metrics}")
    except Exception as exc:  # noqa: BLE001
        print(f"[score error] {variant} seed{seed}: {exc}")


def aggregate(variant: str) -> tuple[dict[str, tuple[float, float]], int]:
    rows = [per_seed_metrics[(variant, s)] for s in SEEDS if (variant, s) in per_seed_metrics]
    if not rows:
        return {}, 0
    keys = set().union(*[set(r.keys()) for r in rows])
    aggregated: dict[str, tuple[float, float]] = {}
    for key in keys:
        values = [r[key] for r in rows if key in r]
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
        aggregated[key] = (mean, std)
    return aggregated, len(rows)


aggregated_metrics = {variant: aggregate(variant) for variant in VARIANTS}
for variant, (agg, n) in aggregated_metrics.items():
    print(f"\n{variant} (n={n} seeds)")
    for key in sorted(agg):
        mean, std = agg[key]
        print(f"  {key:10} {mean:.4f} ± {std:.4f}")

## 10. Results

Final comparison: rows = variants, columns = metrics (CIDEr first), cells =
mean ± std across seeds. Also writes the aggregated + per-seed numbers to
`outputs/caption_eval/results_summary.json`.

In [ ]:
import pandas as pd

METRIC_ORDER = ["CIDEr", "Bleu_4", "METEOR", "ROUGE_L", "SPICE"]

present_metrics = [
    m for m in METRIC_ORDER
    if any(m in per_seed_metrics.get((v, s), {}) for v in VARIANTS for s in SEEDS)
]

table_rows: dict[str, dict[str, str]] = {}
seed_counts: dict[str, int] = {}
for variant in VARIANTS:
    agg, n = aggregated_metrics[variant]
    seed_counts[variant] = n
    table_rows[variant] = {
        metric: (f"{agg[metric][0]:.4f} ± {agg[metric][1]:.4f}" if metric in agg else "—")
        for metric in present_metrics
    }

comparison = pd.DataFrame(table_rows).T
comparison = comparison[present_metrics] if present_metrics else comparison
comparison.index.name = "variant"

print("Seeds aggregated per variant:", seed_counts)
print("Metric backend:", "PTBTokenizer + METEOR/SPICE" if JAVA_OK else "simple tokenizer (BLEU/ROUGE/CIDEr only)")

summary_payload = {
    "variants": VARIANTS,
    "seeds": SEEDS,
    "metrics": present_metrics,
    "seed_counts": seed_counts,
    "java_metrics_enabled": bool(JAVA_OK),
    "aggregated": {
        variant: {m: {"mean": agg[m][0], "std": agg[m][1]} for m in agg}
        for variant, (agg, _n) in aggregated_metrics.items()
    },
    "per_seed": {
        f"{variant}_seed{seed}": per_seed_metrics[(variant, seed)]
        for variant in VARIANTS
        for seed in SEEDS
        if (variant, seed) in per_seed_metrics
    },
}
(CACHE_DIR / "results_summary.json").write_text(json.dumps(summary_payload, indent=2))
print("wrote", CACHE_DIR / "results_summary.json")

comparison